In [ ]:
"""
Pipelines de retraitement pour les flags critiques de la Stratégie 2

Pipeline 1: Retraitement des FLAG_CRITIQUE_NEGATIF (Test 3)
    - Régénère uniquement les variantes_ingredients pour les recettes où
      la variante_ingredients a moins d'actions que la variante_principale

Pipeline 2: Retraitement des FLAG_NO_PRINCIPALE (Test 6)
    - Génère les 3 variantes complètes pour les recettes sans variante_principale
    - Nettoie les résultats avec data_cleaning_3stages
    - Ajoute les résultats au dataset de graphs nettoyés
"""

import pandas as pd
import json
import re
import time
import os
from typing import List, Dict, Tuple
from datetime import datetime
from openai import OpenAI
import numpy as np
import ast


# ==============================================================================
# FONCTIONS UTILITAIRES COMMUNES
# ==============================================================================

def extract_json_from_response(response_text: str) -> dict:
    """
    Extrait le JSON d'une réponse LLM qui peut être mal formatée
    """
    try:
        return json.loads(response_text.strip())
    except json.JSONDecodeError:
        pass
    
    try:
        cleaned_text = response_text.strip()
        if cleaned_text.startswith('```json'):
            cleaned_text = cleaned_text[7:]
        elif cleaned_text.startswith('```'):
            cleaned_text = cleaned_text[3:]
        if cleaned_text.endswith('```'):
            cleaned_text = cleaned_text[:-3]
        cleaned_text = cleaned_text.strip()
        cleaned_text = re.sub(r'/\*.*?\*/', '', cleaned_text, flags=re.DOTALL)
        cleaned_text = re.sub(r'//.*?$', '', cleaned_text, flags=re.MULTILINE)
        return json.loads(cleaned_text)
    except json.JSONDecodeError:
        pass
    
    try:
        markdown_pattern = r'```(?:json)?\s*(.*?)\s*```'
        markdown_match = re.search(markdown_pattern, response_text, re.DOTALL)
        if markdown_match:
            json_text = markdown_match.group(1).strip()
            json_text = re.sub(r'/\*.*?\*/', '', json_text, flags=re.DOTALL)
            json_text = re.sub(r'//.*?$', '', json_text, flags=re.MULTILINE)
            return json.loads(json_text)
    except json.JSONDecodeError:
        pass
    
    try:
        json_pattern = r'\{.*\}'
        json_match = re.search(json_pattern, response_text, re.DOTALL)
        if json_match:
            json_text = json_match.group()
            json_text = re.sub(r'/\*.*?\*/', '', json_text, flags=re.DOTALL)
            json_text = re.sub(r'//.*?$', '', json_text, flags=re.MULTILINE)
            return json.loads(json_text)
    except json.JSONDecodeError:
        pass
    
    return None


def parse_actions(actions_value) -> list:
    """Parse la colonne actions en liste"""
    if pd.isna(actions_value):
        return []
    if isinstance(actions_value, list):
        return actions_value
    if isinstance(actions_value, str):
        try:
            parsed = ast.literal_eval(actions_value)
            if isinstance(parsed, list):
                return parsed
        except (ValueError, SyntaxError):
            if ',' in actions_value:
                return [a.strip() for a in actions_value.split(',') if a.strip()]
            return [actions_value.strip()] if actions_value.strip() else []
    return []


def save_intermediate_results(results: dict, output_dir: str, batch_num: int, pipeline_name: str):
    """Sauvegarde intermédiaire des résultats"""
    os.makedirs(output_dir, exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"{pipeline_name}_intermediate_batch_{batch_num}_{timestamp}.json"
    filepath = os.path.join(output_dir, filename)
    
    with open(filepath, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    
    print(f"💾 Sauvegarde intermédiaire: {filepath}")


# ==============================================================================
# PIPELINE 1: RETRAITEMENT DES FLAG_CRITIQUE_NEGATIF (Test 3)
# Régénère uniquement les variantes_ingredients
# ==============================================================================

def create_prompt_ingredients_variant_optimized(recipes_batch: List[dict]) -> str:
    """
    Crée un prompt OPTIMISÉ pour régénérer les variantes_ingredients
    
    Ce prompt est spécifiquement conçu pour :
    - Analyser les ingrédients pré-transformés
    - Ajouter les gestes de préparation AU DÉBUT de la séquence principale
    - Garantir que la variante_ingredients >= variante_principale en taille
    """
    
    recipes_data = []
    for i, recipe in enumerate(recipes_batch):
        recipe_id = recipe['id']
        actions = recipe['actions']
        actions_str = ", ".join(actions) if isinstance(actions, list) else str(actions)
        ingredients = recipe.get('ingredients', '')
        title = recipe.get('title', f'Recipe_{recipe_id}')
        
        recipe_info = f"""
RECIPE {i+1}:
ID: {recipe_id}
TITLE: {title}
MAIN SEQUENCE: [{actions_str}]
INGREDIENTS: {ingredients}
"""
        recipes_data.append(recipe_info)
    
    recipes_text = "\n".join(recipes_data)
    
    prompt = f"""You are a culinary expert specialized in analyzing ingredient preparation.

Here are {len(recipes_batch)} recipes with their main action sequences:

{recipes_text}

TASK: GENERATE INGREDIENT-BASED VARIANTS

For EACH recipe, analyze the ingredients list and create an ENRICHED variant by adding preparation gestures.

CRITICAL RULES:
1. The ingredient variant MUST contain ALL actions from the main sequence
2. The ingredient variant MUST be LONGER OR EQUAL to the main sequence (NEVER shorter)
3. Add preparation gestures AT THE BEGINNING based on pre-processed ingredients
4. If no pre-processed ingredients found, return the main sequence unchanged

PRE-PROCESSING DETECTION:
Look for these indicators in ingredients and add the corresponding gesture:
- "diced", "diced onions" → add "dice"
- "chopped", "chopped parsley" → add "chop"
- "sliced", "sliced tomatoes" → add "slice"
- "minced", "minced garlic" → add "mince"
- "grated", "grated cheese" → add "grate"
- "shredded", "shredded carrots" → add "shred"
- "cubed", "cubed potatoes" → add "cube"
- "crushed", "crushed crackers" → add "crush"
- "beaten", "beaten eggs" → add "beat"
- "peeled", "peeled apples" → add "peel"
- "cored", "cored apples" → add "core"
- "seeded", "seeded peppers" → add "seed"
- "julienned", "julienned vegetables" → add "julienne"
- "quartered", "quartered mushrooms" → add "quarter"
- "halved", "halved grapes" → add "halve"

STRUCTURE OF INGREDIENT VARIANT:
[preparation gestures from ingredients] + [all actions from main sequence]

EXAMPLE:
- Main sequence: ["mix", "pour", "bake"]
- Ingredients: "diced onions, grated cheese, sliced tomatoes, flour, eggs"
- Pre-processed detected: diced → dice, grated → grate, sliced → slice
- Ingredient variant: ["dice", "grate", "slice", "mix", "pour", "bake"]

RESPONSE FORMAT:
{{
  "recipes": [
    {{
      "id": "recipe_id_1",
      "ingredient_variant": ["action1", "action2", "action3", ...]
    }},
    {{
      "id": "recipe_id_2", 
      "ingredient_variant": ["action1", "action2", ...]
    }}
  ]
}}

IMPORTANT:
- Process ALL {len(recipes_batch)} recipes
- Return ONLY valid JSON, no explanations
- NEVER return a variant shorter than the main sequence
- If unsure, include the gesture rather than omit it
- Only include PHYSICAL GESTURES (no bake, boil, simmer, etc.)
"""
    
    return prompt


def regenerate_ingredients_variants(
    recipes_batch: List[dict],
    api_key: str,
    model_name: str,
    max_retries: int = 3
) -> Dict[str, list]:
    """
    Régénère les variantes_ingredients pour un batch de recettes
    
    Args:
        recipes_batch: Liste de dictionnaires avec {id, title, actions, ingredients}
        api_key: Clé API OpenRouter
        model_name: Nom du modèle LLM
        max_retries: Nombre de tentatives en cas d'erreur
    
    Returns:
        Dict {recipe_id: ingredient_variant_list}
    """
    
    prompt = create_prompt_ingredients_variant_optimized(recipes_batch)
    
    for retry in range(max_retries):
        try:
            client = OpenAI(
                base_url="https://openrouter.ai/api/v1",
                api_key=api_key,
            )
            
            extra_headers = {
                "HTTP-Referer": "recipe_reprocessing_pipeline",
                "X-Title": "Pipeline 1: Ingredients Variant Regeneration"
            }
            
            completion = client.chat.completions.create(
                extra_headers=extra_headers,
                model=model_name,
                messages=[
                    {
                        "role": "system",
                        "content": "Tu es un expert culinaire spécialisé dans l'analyse des ingrédients et la création de séquences d'actions de cuisine."
                    },
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
                max_tokens=8000,
                temperature=0.1,
            )
            
            llm_response = completion.choices[0].message.content.strip()
            print(f"  Réponse reçue ({len(llm_response)} caractères)")
            
            result = extract_json_from_response(llm_response)
            
            if result is None:
                print(f"  ⚠️ Impossible de parser le JSON (tentative {retry+1}/{max_retries})")
                continue
            
            # Extraire les résultats
            ingredients_results = {}
            recipes_results = result.get("recipes", [])
            
            for recipe_result in recipes_results:
                recipe_id = str(recipe_result.get("id"))
                ingredient_variant = recipe_result.get("ingredient_variant", [])
                ingredients_results[recipe_id] = ingredient_variant
            
            return ingredients_results
            
        except Exception as e:
            if "429" in str(e):
                wait_time = (retry + 1) * 3 * 60
                print(f"  ⚠️ Erreur 429, attente de {wait_time//60} minutes (tentative {retry+1}/{max_retries})...")
                time.sleep(wait_time)
            else:
                print(f"  ❌ Erreur: {str(e)}")
                if retry < max_retries - 1:
                    time.sleep(30)
    
    return {}


def run_pipeline_1_critique_negatif(
    flagged_ids: List[str],
    graphs_brut_df: pd.DataFrame,
    recipe_ingredients_df: pd.DataFrame,
    recipes_df: pd.DataFrame,
    api_key: str,
    model_name: str = "mistralai/mistral-small-3.1-24b-instruct:free",
    batch_size: int = 10,
    output_dir: str = "reprocessing_results"
) -> pd.DataFrame:
    """
    PIPELINE 1: Retraitement des FLAG_CRITIQUE_NEGATIF
    
    Régénère les variantes_ingredients pour les recettes où
    len(variante_ingredients) < len(variante_principale)
    
    Args:
        flagged_ids: Liste des IDs de recettes avec FLAG_CRITIQUE_NEGATIF
        graphs_brut_df: DataFrame des graphs bruts (contient les variantes_principales)
        recipe_ingredients_df: DataFrame des ingrédients
        recipes_df: DataFrame des recettes (pour les titres)
        api_key: Clé API OpenRouter
        model_name: Nom du modèle LLM
        batch_size: Taille des batches
        output_dir: Répertoire de sortie
    
    Returns:
        DataFrame avec les nouvelles variantes_ingredients
    """
    
    print(f"\n{'#'*80}")
    print(f"# PIPELINE 1: RETRAITEMENT FLAG_CRITIQUE_NEGATIF")
    print(f"# Régénération des variantes_ingredients")
    print(f"{'#'*80}\n")
    
    os.makedirs(output_dir, exist_ok=True)
    
    # ========== ÉTAPE 1: Préparation des données ==========
    print("ÉTAPE 1: Préparation des données...")
    print("-" * 60)
    
    # Convertir les IDs en string pour la comparaison
    flagged_ids = [str(id) for id in flagged_ids]
    graphs_brut_df['id'] = graphs_brut_df['id'].astype(str)
    
    # Filtrer les variantes principales des IDs flaggés
    principales = graphs_brut_df[
        (graphs_brut_df['id'].isin(flagged_ids)) & 
        (graphs_brut_df['type_2'] == 'variante_principale')
    ].copy()
    
    print(f"  IDs flaggés: {len(flagged_ids)}")
    print(f"  Variantes principales trouvées: {len(principales)}")
    
    # Préparer les ingrédients par recette
    recipe_ingredients_df['id'] = recipe_ingredients_df['id'].astype(str)
    ingredients_grouped = recipe_ingredients_df.groupby('id')['ingredient'].apply(
        lambda x: ', '.join(x.dropna().astype(str))
    ).reset_index()
    ingredients_dict = dict(zip(ingredients_grouped['id'], ingredients_grouped['ingredient']))
    
    # Préparer les titres
    recipes_df['id'] = recipes_df['id'].astype(str)
    titles_dict = dict(zip(recipes_df['id'], recipes_df['title']))
    
    # Construire la liste des recettes à traiter
    recipes_to_process = []
    for _, row in principales.iterrows():
        recipe_id = str(row['id'])
        recipes_to_process.append({
            'id': recipe_id,
            'title': titles_dict.get(recipe_id, f'Recipe_{recipe_id}'),
            'actions': parse_actions(row['actions']),
            'ingredients': ingredients_dict.get(recipe_id, '')
        })
    
    print(f"  Recettes à traiter: {len(recipes_to_process)}")
    
    # ========== ÉTAPE 2: Traitement par batches ==========
    print(f"\nÉTAPE 2: Traitement par batches (taille={batch_size})...")
    print("-" * 60)
    
    all_results = {}
    total_batches = (len(recipes_to_process) + batch_size - 1) // batch_size
    
    for batch_num in range(total_batches):
        start_idx = batch_num * batch_size
        end_idx = min(start_idx + batch_size, len(recipes_to_process))
        batch = recipes_to_process[start_idx:end_idx]
        
        print(f"\n📦 Batch {batch_num + 1}/{total_batches} ({len(batch)} recettes)")
        
        batch_results = regenerate_ingredients_variants(
            recipes_batch=batch,
            api_key=api_key,
            model_name=model_name,
            max_retries=3
        )
        
        all_results.update(batch_results)
        print(f"  ✅ {len(batch_results)} variantes générées")
        
        # Sauvegarde intermédiaire tous les 10 batches
        if (batch_num + 1) % 10 == 0:
            save_intermediate_results(all_results, output_dir, batch_num + 1, "pipeline1")
        
        # Pause entre les batches
        if batch_num < total_batches - 1:
            time.sleep(2)
    
    # ========== ÉTAPE 3: Construction du DataFrame résultat ==========
    print(f"\nÉTAPE 3: Construction du DataFrame résultat...")
    print("-" * 60)
    
    result_rows = []
    for recipe_id, ingredient_variant in all_results.items():
        title = titles_dict.get(recipe_id, f'Recipe_{recipe_id}')
        result_rows.append({
            'id': recipe_id,
            'title': title,
            'actions': ingredient_variant,
            'type': 'secondaire',
            'type_2': 'variante_ingredients'
        })
    
    result_df = pd.DataFrame(result_rows)
    
    # Sauvegarder le résultat final
    output_file = os.path.join(output_dir, 'pipeline1_new_ingredients_variants.csv')
    result_df.to_csv(output_file, index=False)
    
    print(f"  ✅ Résultats sauvegardés: {output_file}")
    print(f"  Total variantes régénérées: {len(result_df)}")
    
    # ========== RÉSUMÉ ==========
    print(f"\n{'='*80}")
    print(f"PIPELINE 1 TERMINÉ")
    print(f"{'='*80}")
    print(f"  Recettes traitées: {len(recipes_to_process)}")
    print(f"  Variantes générées: {len(result_df)}")
    print(f"  Fichier de sortie: {output_file}")
    
    return result_df


# ==============================================================================
# PIPELINE 2: RETRAITEMENT DES FLAG_NO_PRINCIPALE (Test 6)
# Génère les 3 variantes complètes
# ==============================================================================

def create_prompt_all_variants_optimized(recipes_batch: List[dict]) -> str:
    """
    Crée un prompt OPTIMISÉ pour générer les 3 variantes complètes
    (principale, ingredients, permutation)
    """
    
    recipes_data = []
    for i, recipe in enumerate(recipes_batch):
        recipe_info = f"""
RECIPE {i+1}:
ID: {recipe['id']}
TITLE: {recipe['title']}
INSTRUCTIONS: {recipe['instructions']}
INGREDIENTS: {recipe['ingredients']}
"""
        recipes_data.append(recipe_info)
    
    recipes_text = "\n".join(recipes_data)
    
    prompt = f"""You are a culinary expert specialized in analyzing cooking instructions and creating action sequences.

Here are {len(recipes_batch)} recipes to analyze:

{recipes_text}

TASK: Generate 3 VARIANTS of action sequences for EACH recipe

For EACH recipe, create:
1. **MAIN VARIANT**: Extract ALL action verbs from the instructions in order
2. **INGREDIENT VARIANT**: Main variant + preparation gestures from pre-processed ingredients (at the beginning)
3. **PERMUTATION VARIANT**: Logically reorder the main variant while respecting cooking constraints

ACTION VERBS TO EXTRACT:
- Physical gestures: chop, dice, slice, cut, mince, mix, stir, combine, add, pour, season, blend, whip, fold, spread, grate, peel, wash, clean, drain, strain, cube, beat, core, seed, flip, turn, toss, knead, roll, press, squeeze, scrape, sprinkle, garnish, arrange, layer, wrap, brush, coat
- Cooking actions: bake, boil, simmer, roast, fry, sauté, grill, steam, broil, poach, braise, sear, brown, toast, heat, cool, chill, freeze, warm, rest, marinate, reduce

RULES FOR EACH VARIANT:

**MAIN VARIANT:**
- Extract verbs in the order they appear in instructions
- Include both physical gestures AND cooking actions
- One instruction may have 0-3 actions
- Remove consecutive duplicates

**INGREDIENT VARIANT:**
- Start with preparation gestures from pre-processed ingredients
- Pre-processed indicators: diced→dice, chopped→chop, sliced→slice, minced→mince, grated→grate, shredded→shred, cubed→cube, crushed→crush, beaten→beat, peeled→peel
- Then add ALL actions from main variant
- MUST be >= main variant in length

**PERMUTATION VARIANT:**
- Same actions as main variant but in different logical order
- Respect: preparation → combination → cooking → finishing
- Must be different from main variant
- If no valid permutation possible, return main variant

RESPONSE FORMAT:
{{
  "recipes": [
    {{
      "id": "recipe_id_1",
      "main_variant": ["action1", "action2", "action3"],
      "ingredient_variant": ["prep1", "action1", "action2", "action3"],
      "permutation_variant": ["action2", "action1", "action3"]
    }},
    {{
      "id": "recipe_id_2",
      "main_variant": ["action1", "action2"],
      "ingredient_variant": ["action1", "action2"],
      "permutation_variant": ["action1", "action2"]
    }}
  ]
}}

IMPORTANT:
- Process ALL {len(recipes_batch)} recipes
- Return ONLY valid JSON
- Each recipe MUST have all 3 variants
- ingredient_variant length >= main_variant length
- Remove consecutive duplicate actions
"""
    
    return prompt


def generate_all_variants(
    recipes_batch: List[dict],
    api_key: str,
    model_name: str,
    max_retries: int = 3
) -> Dict[str, dict]:
    """
    Génère les 3 variantes pour un batch de recettes
    
    Returns:
        Dict {recipe_id: {main_variant, ingredient_variant, permutation_variant}}
    """
    
    prompt = create_prompt_all_variants_optimized(recipes_batch)
    
    for retry in range(max_retries):
        try:
            client = OpenAI(
                base_url="https://openrouter.ai/api/v1",
                api_key=api_key,
            )
            
            extra_headers = {
                "HTTP-Referer": "recipe_reprocessing_pipeline",
                "X-Title": "Pipeline 2: Full Variants Generation"
            }
            
            completion = client.chat.completions.create(
                extra_headers=extra_headers,
                model=model_name,
                messages=[
                    {
                        "role": "system",
                        "content": "Tu es un expert culinaire qui analyse des instructions de cuisine et crée des séquences d'actions précises."
                    },
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
                max_tokens=10000,
                temperature=0.1,
            )
            
            llm_response = completion.choices[0].message.content.strip()
            print(f"  Réponse reçue ({len(llm_response)} caractères)")
            
            result = extract_json_from_response(llm_response)
            
            if result is None:
                print(f"  ⚠️ Impossible de parser le JSON (tentative {retry+1}/{max_retries})")
                continue
            
            # Extraire les résultats
            all_variants = {}
            recipes_results = result.get("recipes", [])
            
            for recipe_result in recipes_results:
                recipe_id = str(recipe_result.get("id"))
                all_variants[recipe_id] = {
                    'main_variant': recipe_result.get("main_variant", []),
                    'ingredient_variant': recipe_result.get("ingredient_variant", []),
                    'permutation_variant': recipe_result.get("permutation_variant", [])
                }
            
            return all_variants
            
        except Exception as e:
            if "429" in str(e):
                wait_time = (retry + 1) * 3 * 60
                print(f"  ⚠️ Erreur 429, attente de {wait_time//60} minutes (tentative {retry+1}/{max_retries})...")
                time.sleep(wait_time)
            else:
                print(f"  ❌ Erreur: {str(e)}")
                if retry < max_retries - 1:
                    time.sleep(30)
    
    return {}


def run_pipeline_2_no_principale(
    flagged_ids: List[str],
    recipes_df: pd.DataFrame,
    recipe_instructions_df: pd.DataFrame,
    recipe_ingredients_df: pd.DataFrame,
    api_key: str,
    model_name: str = "mistralai/mistral-small-3.1-24b-instruct:free",
    batch_size: int = 5,
    output_dir: str = "reprocessing_results",
    apply_cleaning: bool = True
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    PIPELINE 2: Retraitement des FLAG_NO_PRINCIPALE
    
    Génère les 3 variantes complètes pour les recettes sans variante_principale,
    puis nettoie les résultats avec data_cleaning_3stages
    
    Args:
        flagged_ids: Liste des IDs de recettes avec FLAG_NO_PRINCIPALE
        recipes_df: DataFrame des recettes
        recipe_instructions_df: DataFrame des instructions
        recipe_ingredients_df: DataFrame des ingrédients
        api_key: Clé API OpenRouter
        model_name: Nom du modèle LLM
        batch_size: Taille des batches (plus petit car génère 3 variantes)
        output_dir: Répertoire de sortie
        apply_cleaning: Appliquer data_cleaning_3stages (nécessite les fichiers de nettoyage)
    
    Returns:
        Tuple (df_cleaned_without_non_gestures, df_cleaned_with_non_gestures)
    """
    
    print(f"\n{'#'*80}")
    print(f"# PIPELINE 2: RETRAITEMENT FLAG_NO_PRINCIPALE")
    print(f"# Génération des 3 variantes complètes")
    print(f"{'#'*80}\n")
    
    os.makedirs(output_dir, exist_ok=True)
    
    # ========== ÉTAPE 1: Préparation des données ==========
    print("ÉTAPE 1: Préparation des données...")
    print("-" * 60)
    
    # Convertir les IDs en string
    flagged_ids = [str(id) for id in flagged_ids]
    recipes_df['id'] = recipes_df['id'].astype(str)
    recipe_instructions_df['id'] = recipe_instructions_df['id'].astype(str)
    recipe_ingredients_df['id'] = recipe_ingredients_df['id'].astype(str)
    
    # Filtrer les recettes concernées
    recipes_filtered = recipes_df[recipes_df['id'].isin(flagged_ids)].copy()
    print(f"  IDs flaggés: {len(flagged_ids)}")
    print(f"  Recettes trouvées: {len(recipes_filtered)}")
    
    # Grouper les instructions par recette
    instructions_grouped = recipe_instructions_df[
        recipe_instructions_df['id'].isin(flagged_ids)
    ].groupby('id')['instruction'].apply(
        lambda x: ' '.join(x.dropna().astype(str))
    ).reset_index()
    instructions_dict = dict(zip(instructions_grouped['id'], instructions_grouped['instruction']))
    
    # Grouper les ingrédients par recette
    ingredients_grouped = recipe_ingredients_df[
        recipe_ingredients_df['id'].isin(flagged_ids)
    ].groupby('id')['ingredient'].apply(
        lambda x: ', '.join(x.dropna().astype(str))
    ).reset_index()
    ingredients_dict = dict(zip(ingredients_grouped['id'], ingredients_grouped['ingredient']))
    
    # Construire la liste des recettes à traiter
    recipes_to_process = []
    for _, row in recipes_filtered.iterrows():
        recipe_id = str(row['id'])
        recipes_to_process.append({
            'id': recipe_id,
            'title': row.get('title', f'Recipe_{recipe_id}'),
            'instructions': instructions_dict.get(recipe_id, ''),
            'ingredients': ingredients_dict.get(recipe_id, '')
        })
    
    # Filtrer les recettes avec instructions non vides
    recipes_to_process = [r for r in recipes_to_process if r['instructions'].strip()]
    print(f"  Recettes avec instructions: {len(recipes_to_process)}")
    
    # ========== ÉTAPE 2: Traitement par batches ==========
    print(f"\nÉTAPE 2: Traitement par batches (taille={batch_size})...")
    print("-" * 60)
    
    all_results = {}
    total_batches = (len(recipes_to_process) + batch_size - 1) // batch_size
    
    for batch_num in range(total_batches):
        start_idx = batch_num * batch_size
        end_idx = min(start_idx + batch_size, len(recipes_to_process))
        batch = recipes_to_process[start_idx:end_idx]
        
        print(f"\n📦 Batch {batch_num + 1}/{total_batches} ({len(batch)} recettes)")
        
        batch_results = generate_all_variants(
            recipes_batch=batch,
            api_key=api_key,
            model_name=model_name,
            max_retries=3
        )
        
        all_results.update(batch_results)
        print(f"  ✅ {len(batch_results)} recettes traitées")
        
        # Sauvegarde intermédiaire tous les 5 batches
        if (batch_num + 1) % 5 == 0:
            save_intermediate_results(all_results, output_dir, batch_num + 1, "pipeline2")
        
        # Pause entre les batches
        if batch_num < total_batches - 1:
            time.sleep(3)
    
    # ========== ÉTAPE 3: Construction du DataFrame résultat ==========
    print(f"\nÉTAPE 3: Construction du DataFrame résultat...")
    print("-" * 60)
    
    # Créer les titres dict
    titles_dict = dict(zip(recipes_df['id'], recipes_df['title']))
    
    result_rows = []
    for recipe_id, variants in all_results.items():
        title = titles_dict.get(recipe_id, f'Recipe_{recipe_id}')
        
        # Variante principale
        result_rows.append({
            'id': recipe_id,
            'title': title,
            'actions': variants.get('main_variant', []),
            'type': 'principal',
            'type_2': 'variante_principale'
        })
        
        # Variante ingrédients
        result_rows.append({
            'id': recipe_id,
            'title': title,
            'actions': variants.get('ingredient_variant', []),
            'type': 'secondaire',
            'type_2': 'variante_ingredients'
        })
        
        # Variante permutation
        result_rows.append({
            'id': recipe_id,
            'title': title,
            'actions': variants.get('permutation_variant', []),
            'type': 'secondaire',
            'type_2': 'variante_permutation'
        })
    
    result_df = pd.DataFrame(result_rows)
    
    # Sauvegarder le résultat brut
    raw_output_file = os.path.join(output_dir, 'pipeline2_new_variants_raw.csv')
    result_df.to_csv(raw_output_file, index=False)
    print(f"  ✅ Résultats bruts sauvegardés: {raw_output_file}")
    
    # ========== ÉTAPE 4: Nettoyage avec data_cleaning_3stages ==========
    if apply_cleaning:
        print(f"\nÉTAPE 4: Application de data_cleaning_3stages...")
        print("-" * 60)
        
        try:
            # Importer la fonction depuis recipe_management
            # Note: Assure-toi que recipe_management.py est dans le même répertoire
            # ou ajuste l'import selon ton setup
            from recipe_management import data_cleaning_3stages
            
            df_without_gestures, df_with_gestures = data_cleaning_3stages(result_df)
            
            # Sauvegarder les résultats nettoyés
            clean_without_file = os.path.join(output_dir, 'pipeline2_cleaned_without_non_gestures.csv')
            clean_with_file = os.path.join(output_dir, 'pipeline2_cleaned_with_non_gestures.csv')
            
            df_without_gestures.to_csv(clean_without_file, index=False)
            df_with_gestures.to_csv(clean_with_file, index=False)
            
            print(f"  ✅ Données nettoyées (sans non-gestes): {clean_without_file}")
            print(f"  ✅ Données nettoyées (avec non-gestes): {clean_with_file}")
            
        except ImportError as e:
            print(f"  ⚠️ Impossible d'importer data_cleaning_3stages: {e}")
            print(f"  Les données brutes sont disponibles dans: {raw_output_file}")
            df_without_gestures = result_df
            df_with_gestures = result_df
        except Exception as e:
            print(f"  ⚠️ Erreur lors du nettoyage: {e}")
            df_without_gestures = result_df
            df_with_gestures = result_df
    else:
        df_without_gestures = result_df
        df_with_gestures = result_df
    
    # ========== RÉSUMÉ ==========
    print(f"\n{'='*80}")
    print(f"PIPELINE 2 TERMINÉ")
    print(f"{'='*80}")
    print(f"  Recettes traitées: {len(recipes_to_process)}")
    print(f"  Variantes générées: {len(result_df)} ({len(result_df)//3} recettes × 3 variantes)")
    print(f"  Après nettoyage (sans non-gestes): {len(df_without_gestures)} variantes")
    print(f"  Après nettoyage (avec non-gestes): {len(df_with_gestures)} variantes")
    
    return df_without_gestures, df_with_gestures


# ==============================================================================
# FONCTION PRINCIPALE D'ORCHESTRATION
# ==============================================================================

def run_all_reprocessing_pipelines(
    dataset_synthese_path: str,
    graphs_brut_path: str,
    recipes_path: str,
    recipe_instructions_path: str,
    recipe_ingredients_path: str,
    api_key: str,
    model_name: str = "mistralai/mistral-small-3.1-24b-instruct:free",
    output_dir: str = "reprocessing_results"
):
    """
    Exécute les deux pipelines de retraitement
    
    Args:
        dataset_synthese_path: Chemin vers dataset_synthese.csv (contient les flags)
        graphs_brut_path: Chemin vers graphs_recipes.csv (brut)
        recipes_path: Chemin vers recipes.csv
        recipe_instructions_path: Chemin vers recipe_instructions.csv
        recipe_ingredients_path: Chemin vers recipe_ingredients.csv
        api_key: Clé API OpenRouter
        model_name: Nom du modèle LLM
        output_dir: Répertoire de sortie
    """
    
    print(f"\n{'#'*80}")
    print(f"# EXÉCUTION DES PIPELINES DE RETRAITEMENT")
    print(f"{'#'*80}\n")
    
    # Charger les données
    print("Chargement des données...")
    dataset_synthese = pd.read_csv(dataset_synthese_path)
    graphs_brut_df = pd.read_csv(graphs_brut_path)
    recipes_df = pd.read_csv(recipes_path)
    recipe_instructions_df = pd.read_csv(recipe_instructions_path)
    recipe_ingredients_df = pd.read_csv(recipe_ingredients_path)
    
    # Extraire les IDs flaggés
    ids_test3_critique = dataset_synthese[
        (dataset_synthese['test'] == '3') & 
        (dataset_synthese['flag'] == 'FLAG_CRITIQUE_NEGATIF')
    ]['id'].unique().tolist()
    
    ids_test6_no_principale = dataset_synthese[
        (dataset_synthese['test'] == '6') & 
        (dataset_synthese['flag'] == 'FLAG_NO_PRINCIPALE')
    ]['id'].unique().tolist()
    
    print(f"IDs FLAG_CRITIQUE_NEGATIF (Test 3): {len(ids_test3_critique)}")
    print(f"IDs FLAG_NO_PRINCIPALE (Test 6): {len(ids_test6_no_principale)}")
    
    # Pipeline 1: FLAG_CRITIQUE_NEGATIF
    if len(ids_test3_critique) > 0:
        print(f"\n{'='*80}")
        print("Exécution du Pipeline 1...")
        result_pipeline1 = run_pipeline_1_critique_negatif(
            flagged_ids=ids_test3_critique,
            graphs_brut_df=graphs_brut_df,
            recipe_ingredients_df=recipe_ingredients_df,
            recipes_df=recipes_df,
            api_key=api_key,
            model_name=model_name,
            batch_size=10,
            output_dir=output_dir
        )
    
    # Pipeline 2: FLAG_NO_PRINCIPALE
    if len(ids_test6_no_principale) > 0:
        print(f"\n{'='*80}")
        print("Exécution du Pipeline 2...")
        result_pipeline2_without, result_pipeline2_with = run_pipeline_2_no_principale(
            flagged_ids=ids_test6_no_principale,
            recipes_df=recipes_df,
            recipe_instructions_df=recipe_instructions_df,
            recipe_ingredients_df=recipe_ingredients_df,
            api_key=api_key,
            model_name=model_name,
            batch_size=5,
            output_dir=output_dir,
            apply_cleaning=True
        )
    
    print(f"\n{'#'*80}")
    print(f"# TOUS LES PIPELINES TERMINÉS")
    print(f"{'#'*80}")
    print(f"Résultats disponibles dans: {output_dir}/")




In [ ]:
# ==============================================================================
# EXEMPLE D'UTILISATION
# ==============================================================================

if __name__ == "__main__":
    """
    Exemple d'utilisation des pipelines de retraitement
    """
    import os
    # Charger les variables du fichier .env
    load_dotenv()

    OPENROUTER_API_KEY = os.getenv('OPENROUTER_API_KEY_1')
    MODEL_NAME = os.getenv('MODEL_NAME_2')
    
    # Chemins des fichiers
    DATASET_SYNTHESE = "strategy_2_results/dataset_synthese.csv"
    GRAPHS_BRUT = "graphs_recipes.csv"
    RECIPES = "recipes.csv"
    RECIPE_INSTRUCTIONS = "recipe_instructions.csv"
    RECIPE_INGREDIENTS = "recipe_ingredients.csv"
    
    # Exécuter les pipelines
    run_all_reprocessing_pipelines(
        dataset_synthese_path=DATASET_SYNTHESE,
        graphs_brut_path=GRAPHS_BRUT,
        recipes_path=RECIPES,
        recipe_instructions_path=RECIPE_INSTRUCTIONS,
        recipe_ingredients_path=RECIPE_INGREDIENTS,
        api_key=OPENROUTER_API_KEY,
        model_name=MODEL_NAME,
        output_dir="reprocessing_results"
    )